# NLP Practical Assignment – Sentiment Analysis
### IMDB Movie Reviews | RNN, LSTM & GRU

**Environment:** Jupyter Notebook  
**Framework:** TensorFlow / Keras  
**Goal:** Build, train, evaluate, compare, and test three recurrent neural-network models for movie-review sentiment classification.

> Run the notebook from top to bottom. Training all three models may take some time, especially without a GPU.

## Table of Contents

1. Data Preparation (Tasks 1–7)
2. Simple RNN (Tasks 8–14)
3. LSTM & GRU (Tasks 15–18)
4. Model Comparison (Tasks 19–20)
5. Real Predictions (Tasks 21–22)
6. Error Analysis (Tasks 23–24)

## Phase 1 : Data Preparation
### Task 1 – Import Libraries
Import the libraries required for data handling, visualization, model building, training, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import re

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("All libraries imported successfully!")

### Task 2 – Configuration

Create the project configuration variables.

In [ ]:
VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128
RNN_UNITS = 64
BATCH_SIZE = 64
EPOCHS = 10
RANDOM_STATE = 42

print("VOCAB_SIZE:", VOCAB_SIZE)
print("MAX_LENGTH:", MAX_LENGTH)
print("EMBEDDING_DIM:", EMBEDDING_DIM)
print("RNN_UNITS:", RNN_UNITS)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("RANDOM_STATE:", RANDOM_STATE)

### Task 3 – Load Dataset

Load the IMDB dataset from Keras. The dataset is already divided into training and testing sets.

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

### Task 4 – Explore Dataset

First, display the dataset in a table-like format. The original Keras dataset contains integer-encoded reviews rather than raw text.

The second DataFrame below decodes the first five reviews so that they are easier to read.

In [ ]:
# Display the first five reviews as encoded word IDs
encoded_df = pd.DataFrame({
    "Review": list(X_train[:5]),
    "Sentiment": y_train[:5]
})

display(encoded_df.head())

In [ ]:
# Create the reverse word index before decoding reviews
word_index = imdb.get_word_index()

reverse_word_index = {
    value + 3: key
    for key, value in word_index.items()
}

reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"


def decode_review(encoded_review):
    decoded_words = []

    for word_id in encoded_review:
        word = reverse_word_index.get(word_id, "<UNK>")
        decoded_words.append(word)

    return " ".join(decoded_words)


# Display the first five reviews as readable text
readable_df = pd.DataFrame({
    "Review": [decode_review(review) for review in X_train[:5]],
    "Sentiment": y_train[:5]
})

display(readable_df.head())

In [ ]:
print("First review (encoded):")
print(X_train[0])

print("\nFirst review (decoded):")
print(decode_review(X_train[0]))

print("\nFirst review label:", y_train[0])

positive_reviews = np.sum(y_train == 1)
negative_reviews = np.sum(y_train == 0)

print("\nPositive reviews:", positive_reviews)
print("Negative reviews:", negative_reviews)

### Task 5 – Review Length

Calculate the number of words (encoded tokens) in each training review and visualize the distribution.

In [ ]:
review_lengths = [len(review) for review in X_train]

print("First 10 review lengths:")
print(review_lengths[:10])

plt.figure(figsize=(10, 5))
plt.hist(review_lengths, bins=50)
plt.title("IMDB Review Length Distribution")
plt.xlabel("Review Length")
plt.ylabel("Number of Reviews")
plt.show()

### Task 6 – Decode Reviews

The `decode_review()` function was created in Task 4 because it is also needed to display readable reviews in the dataset exploration step.

Try it on another review:

In [ ]:
print(decode_review(X_train[1]))

### Task 7 – Padding

Make every review exactly `MAX_LENGTH` tokens long. Short reviews are padded at the end; long reviews are truncated at the end.

In [ ]:
print("First review length before padding:", len(X_train[0]))

X_train_pad = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("First review length after padding:", len(X_train_pad[0]))
print("X_train_pad shape:", X_train_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

##  Phase 2 : Build & Train Models
### Task 8 – Build Simple RNN

Create a function that builds and compiles the SimpleRNN model.

In [ ]:
def build_rnn_model():
    model = Sequential()

    model.add(Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ))

    model.add(SimpleRNN(RNN_UNITS))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Task 9 – Model Summary

Create the RNN model and display its architecture.

In [ ]:
rnn_model = build_rnn_model()
rnn_model.summary()

### Task 10 – Early Stopping

Create a fresh EarlyStopping callback for each model training run. This helps stop training when validation loss stops improving.

In [ ]:
def create_early_stopping():
    return EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )

### Task 11 – Train RNN

Train the RNN model and measure its training time.

In [ ]:
start_time = time.time()

rnn_history = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[create_early_stopping()]
)

rnn_training_time = time.time() - start_time

print("RNN training time:", round(rnn_training_time, 2), "seconds")

### Task 12 – Evaluate RNN

Predict the test labels, calculate evaluation metrics, and print the classification report.

In [ ]:
rnn_probabilities = rnn_model.predict(X_test_pad, verbose=0)
rnn_predictions = (rnn_probabilities >= 0.5).astype(int).flatten()

rnn_accuracy = accuracy_score(y_test, rnn_predictions)
rnn_precision = precision_score(y_test, rnn_predictions, zero_division=0)
rnn_recall = recall_score(y_test, rnn_predictions, zero_division=0)
rnn_f1 = f1_score(y_test, rnn_predictions, zero_division=0)

print("Accuracy:", rnn_accuracy)
print("Precision:", rnn_precision)
print("Recall:", rnn_recall)
print("F1 Score:", rnn_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    rnn_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

### Task 13 – Confusion Matrix

Visualize the RNN model's correct and incorrect predictions.

In [ ]:
rnn_cm = confusion_matrix(y_test, rnn_predictions)

rnn_display = ConfusionMatrixDisplay(
    confusion_matrix=rnn_cm,
    display_labels=["Negative", "Positive"]
)

rnn_display.plot(cmap="Blues")
plt.title("RNN Confusion Matrix")
plt.show()

### Task 14 – Learning Curves

Plot training and validation accuracy, followed by training and validation loss.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(rnn_history.history["accuracy"], label="Training Accuracy")
plt.plot(rnn_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("RNN Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(rnn_history.history["loss"], label="Training Loss")
plt.plot(rnn_history.history["val_loss"], label="Validation Loss")
plt.title("RNN Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Phase 3 : LSTM & GRU
### Task 15 – Build LSTM

Create an LSTM model with the same architecture as the RNN model, replacing `SimpleRNN` with `LSTM`.

In [ ]:
def build_lstm_model():
    model = Sequential()

    model.add(Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ))

    model.add(LSTM(RNN_UNITS))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Task 16 – Train & Evaluate LSTM

Build, summarize, train, and evaluate the LSTM model. Then display its confusion matrix and learning curves.

In [ ]:
lstm_model = build_lstm_model()
lstm_model.summary()

In [ ]:
start_time = time.time()

lstm_history = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[create_early_stopping()]
)

lstm_training_time = time.time() - start_time

print("LSTM training time:", round(lstm_training_time, 2), "seconds")

In [ ]:
lstm_probabilities = lstm_model.predict(X_test_pad, verbose=0)
lstm_predictions = (lstm_probabilities >= 0.5).astype(int).flatten()

lstm_accuracy = accuracy_score(y_test, lstm_predictions)
lstm_precision = precision_score(y_test, lstm_predictions, zero_division=0)
lstm_recall = recall_score(y_test, lstm_predictions, zero_division=0)
lstm_f1 = f1_score(y_test, lstm_predictions, zero_division=0)

print("Accuracy:", lstm_accuracy)
print("Precision:", lstm_precision)
print("Recall:", lstm_recall)
print("F1 Score:", lstm_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    lstm_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

In [ ]:
lstm_cm = confusion_matrix(y_test, lstm_predictions)

lstm_display = ConfusionMatrixDisplay(
    confusion_matrix=lstm_cm,
    display_labels=["Negative", "Positive"]
)

lstm_display.plot(cmap="Blues")
plt.title("LSTM Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lstm_history.history["accuracy"], label="Training Accuracy")
plt.plot(lstm_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lstm_history.history["loss"], label="Training Loss")
plt.plot(lstm_history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Task 17 – Build GRU

Create a GRU model with the same architecture, replacing the recurrent layer with `GRU`.

In [ ]:
def build_gru_model():
    model = Sequential()

    model.add(Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ))

    model.add(GRU(RNN_UNITS))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Task 18 – Train & Evaluate GRU

Build, summarize, train, and evaluate the GRU model. Then display its confusion matrix and learning curves.

In [ ]:
gru_model = build_gru_model()
gru_model.summary()

In [ ]:
start_time = time.time()

gru_history = gru_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[create_early_stopping()]
)

gru_training_time = time.time() - start_time

print("GRU training time:", round(gru_training_time, 2), "seconds")

In [ ]:
gru_probabilities = gru_model.predict(X_test_pad, verbose=0)
gru_predictions = (gru_probabilities >= 0.5).astype(int).flatten()

gru_accuracy = accuracy_score(y_test, gru_predictions)
gru_precision = precision_score(y_test, gru_predictions, zero_division=0)
gru_recall = recall_score(y_test, gru_predictions, zero_division=0)
gru_f1 = f1_score(y_test, gru_predictions, zero_division=0)

print("Accuracy:", gru_accuracy)
print("Precision:", gru_precision)
print("Recall:", gru_recall)
print("F1 Score:", gru_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    gru_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

In [ ]:
gru_cm = confusion_matrix(y_test, gru_predictions)

gru_display = ConfusionMatrixDisplay(
    confusion_matrix=gru_cm,
    display_labels=["Negative", "Positive"]
)

gru_display.plot(cmap="Blues")
plt.title("GRU Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gru_history.history["accuracy"], label="Training Accuracy")
plt.plot(gru_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("GRU Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gru_history.history["loss"], label="Training Loss")
plt.plot(gru_history.history["val_loss"], label="Validation Loss")
plt.title("GRU Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Phase 4 : Comparison
### Task 19 – Compare All Models

Create a Pandas DataFrame containing the evaluation metrics and training time for all three models.

In [ ]:
results = {
    "Model": ["RNN", "LSTM", "GRU"],
    "Accuracy": [rnn_accuracy, lstm_accuracy, gru_accuracy],
    "Precision": [rnn_precision, lstm_precision, gru_precision],
    "Recall": [rnn_recall, lstm_recall, gru_recall],
    "F1": [rnn_f1, lstm_f1, gru_f1],
    "Training Time": [
        rnn_training_time,
        lstm_training_time,
        gru_training_time
    ]
}

results_df = pd.DataFrame(results)

display(results_df)

### Task 20 – Visualization

Create separate bar charts to compare the models across each metric.

#### Models Accuracy Comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.title("Models Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.show()

#### Models Precision Comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results_df["Model"], results_df["Precision"])
plt.title("Models Precision Comparison")
plt.xlabel("Model")
plt.ylabel("Precision")
plt.show()

#### Models Recall Comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results_df["Model"], results_df["Recall"])
plt.title("Models Recall Comparison")
plt.xlabel("Model")
plt.ylabel("Recall")
plt.show()

#### Models F1 Score Comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results_df["Model"], results_df["F1"])
plt.title("Models F1 Score Comparison")
plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.show()

#### Models Training Time Comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results_df["Model"], results_df["Training Time"])
plt.title("Models Training Time Comparison")
plt.xlabel("Model")
plt.ylabel("Time (Seconds)")
plt.show()

In [ ]:
best_f1_model = results_df.loc[results_df["F1"].idxmax(), "Model"]
fastest_model = results_df.loc[results_df["Training Time"].idxmin(), "Model"]
best_accuracy_model = results_df.loc[results_df["Accuracy"].idxmax(), "Model"]

print("Best F1 model:", best_f1_model)
print("Fastest model:", fastest_model)
print("Best Accuracy model:", best_accuracy_model)

## Phase 5 : Real Predictions
### Task 21 – Encode New Review

Convert a new text review into word IDs using the IMDB word index, handle unknown/out-of-vocabulary words, and pad it to `MAX_LENGTH`.

**Important:** Keras IMDB uses reserved IDs: `1` for the start token and `2` for unknown words. The word-index values are shifted by 3.

In [ ]:
def encode_new_review(text):
    text = text.lower()
    words = re.findall(r"\b\w+\b", text)

    encoded_review = [1]  # Start token

    for word in words:
        word_id = word_index.get(word)

        if word_id is None:
            encoded_review.append(2)  # Unknown word
        else:
            word_id = word_id + 3

            if word_id >= VOCAB_SIZE:
                word_id = 2  # Out-of-vocabulary word
            encoded_review.append(word_id)

    padded_review = pad_sequences(
        [encoded_review],
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    return padded_review


sample_review = "This movie was absolutely amazing"
encoded_sample = encode_new_review(sample_review)

print("Encoded review:")
print(encoded_sample)
print("Shape:", encoded_sample.shape)

### Task 22 – Prediction Function

Create a function that predicts the sentiment and reports the model's confidence for its selected class. Then test all four reviews using RNN, LSTM, and GRU.

In [ ]:
def predict_sentiment(model, text):
    encoded_review = encode_new_review(text)

    probability = float(
        model.predict(encoded_review, verbose=0)[0][0]
    )

    if probability >= 0.5:
        sentiment = "Positive"
        confidence = probability * 100
    else:
        sentiment = "Negative"
        confidence = (1 - probability) * 100

    return {
        "Sentiment": sentiment,
        "Confidence": round(confidence, 2)
    }

In [ ]:
reviews = [
    "This movie was absolutely amazing and I loved it",
    "This movie was terrible and boring",
    "I really enjoyed this film",
    "The movie was not good and I hated it"
]

models = {
    "RNN": rnn_model,
    "LSTM": lstm_model,
    "GRU": gru_model
}

for review in reviews:
    print("\nReview:", review)

    for model_name, model in models.items():
        result = predict_sentiment(model, review)

        print(
            model_name,
            "->",
            result["Sentiment"],
            "| Confidence:",
            result["Confidence"],
            "%"
        )

## Phase 6 : Error Analysis
### Task 23 – Find Wrong Predictions

Compare GRU predictions with the true test labels and find the indices of incorrect predictions.

In [ ]:
wrong_indices = np.where(gru_predictions != y_test)[0]

print("Number of wrong predictions:", len(wrong_indices))
print("First 10 wrong indices:", wrong_indices[:10])

### Task 24 – Display Errors

Display the first ten incorrect predictions with the decoded review, actual sentiment, and predicted sentiment.

In [ ]:
for index in wrong_indices[:10]:
    review_text = decode_review(X_test[index])

    actual = y_test[index]
    predicted = gru_predictions[index]

    actual_sentiment = "Positive" if actual == 1 else "Negative"
    predicted_sentiment = "Positive" if predicted == 1 else "Negative"

    print("\n" + "-" * 70)
    print("Review Text:")
    print(review_text)
    print("\nActual Sentiment:", actual_sentiment)
    print("Predicted Sentiment:", predicted_sentiment)
    print("-" * 70)

## Final Challenge – Check Your Understanding

Try to answer these questions without looking at the explanations:

1. Why do we use an Embedding layer?
2. What is the difference between Padding and Truncating?
3. Why can a SimpleRNN struggle to remember older information?
4. What is the difference between LSTM and GRU?
5. Why did we use Sigmoid in the output layer?
6. What is the difference between Accuracy and F1 Score?
7. Why do we use EarlyStopping?
8. How can we recognize overfitting from the learning curves?
9. Why do we perform Error Analysis?
10. If GRU has the highest F1 score but LSTM trains faster, which would you choose and why?

---

## Important Notes

- Run cells in order, from top to bottom.
- The first five decoded reviews are shown in Task 4 to help you inspect the dataset before modeling.
- The test set is used for final evaluation. For a stricter model-selection workflow, use a separate validation set to choose models and reserve the test set for the final comparison.
- Training time and metric values depend on your hardware and training run.
- The confidence value is the model's probability for its predicted class; it is not a guarantee that the prediction is correct.